In [ ]:
import os
import h5py
import numpy as np
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax

# Try importing TeNPy helper for native MPS deserialization
try:
    import tenpy.tools.hdf5_io as h5_tenpy
    from tenpy.algorithms.exact_diag import get_full_wavefunction
    HAS_TENPY = True
except ImportError:
    HAS_TENPY = False


# ==========================================
# 1. KAGOME ADJACENCY MATRIX GENERATOR
# ==========================================
def build_kagome_adjacency(Lx: int, Ly: int, periodic: bool = True) -> jnp.ndarray:
    """
    Builds the exact 2D Kagome lattice adjacency matrix.
    N = 3 * Lx * Ly total lattice sites across 3-site unit cells (A, B, C).
    """
    num_unit_cells = Lx * Ly
    num_sites = 3 * num_unit_cells
    adj = np.zeros((num_sites, num_sites), dtype=np.float32)

    def get_site_index(x, y, sublattice):
        if periodic:
            x = x % Lx
            y = y % Ly
        else:
            if x < 0 or x >= Lx or y < 0 or y >= Ly:
                return -1
        uc_index = x + y * Lx
        return 3 * uc_index + sublattice

    def add_edge(site1, site2):
        if site1 != -1 and site2 != -1:
            adj[site1, site2] = 1.0
            adj[site2, site1] = 1.0

    for y in range(Ly):
        for x in range(Lx):
            A = get_site_index(x, y, 0)
            B = get_site_index(x, y, 1)
            C = get_site_index(x, y, 2)

            # Intra-unit cell bonds (internal triangle)
            add_edge(A, B)
            add_edge(B, C)
            add_edge(C, A)

            # Inter-unit cell bonds
            B_left = get_site_index(x - 1, y, 1)
            add_edge(A, B_left)

            A_down = get_site_index(x, y - 1, 0)
            add_edge(C, A_down)

            C_diag = get_site_index(x - 1, y - 1, 2)
            add_edge(B, C_diag)

    return jnp.array(adj)


# ==========================================
# 2. HDF5 DATASET LOADER
# ==========================================
def load_kagome_dataset(h5_filepath):
    print(f"--- Loading Kagome Dataset from '{h5_filepath}' ---")

    if not os.path.exists(h5_filepath):
        raise FileNotFoundError(f"Could not locate file: {h5_filepath}")

    with h5py.File(h5_filepath, "r") as f:
        p = f["model_params"]
        Lx = int(p["Lx"][()]) if "Lx" in p else 1
        Ly = int(p["Ly"][()]) if "Ly" in p else 1
        num_elec = int(p["Num_Elec"][()]) if "Num_Elec" in p else 1
        U = float(p["U"][()]) if "U" in p else 0.0
        t1 = float(p["t1"][()]) if "t1" in p else 1.0
        t2 = float(p["t2"][()]) if "t2" in p else 0.0
        theta_x = float(p["theta_x"][()]) if "theta_x" in p else 0.0
        theta_y = float(p["theta_y"][()]) if "theta_y" in p else 0.0

        param_vector = np.array([U, t1, t2, theta_x, theta_y], dtype=np.float32)

        obs = f["observables"]

        def get_obs(key):
            if key in obs:
                return np.array(obs[key], dtype=np.float32)
            raise KeyError(f"Required observable '{key}' not found in HDF5 file.")

        density_true = np.squeeze(get_obs("charge_density"))
        spin_corr_true = np.squeeze(get_obs("spin_corr"))
        charge_corr_true = np.squeeze(get_obs("charge_corr"))
        double_occupancy_true = np.squeeze(get_obs("double_occupancy"))
        spin_s_q_true = np.squeeze(get_obs("S_q_spin"))
        charge_s_q_true = np.squeeze(get_obs("S_q_charge"))

        num_sites = len(density_true)

        observables = {
            "charge_density": density_true,
            "spin_corr": spin_corr_true,
            "charge_corr": charge_corr_true,
            "double_occupancy": double_occupancy_true,
            #"S_q_spin": spin_s_q_true,
            #"S_q_charge": charge_s_q_true
        }

        # Wavefunction Deserialization
        y_psi = None
        if "psi" in f and HAS_TENPY:
            psi_mps = h5_tenpy.load_from_hdf5(f, "psi")
            psi_dense = get_full_wavefunction(psi_mps)
            y_psi = np.array(psi_dense, dtype=np.float32).reshape(-1)
        elif "y_psi" in f:
            y_psi = np.array(f["y_psi"], dtype=np.float32)

    # Build spin basis configuration matrix
    if y_psi is not None:
        dim = len(y_psi)
        y_psi = y_psi / np.linalg.norm(y_psi)

        basis_indices = np.arange(dim)
        binary_matrix = (basis_indices[:, None] >> np.arange(num_sites)[::-1]) & 1
        X_data = (2 * binary_matrix - 1).astype(np.float32)
    else:
        X_data = np.repeat(param_vector[None, :], 16, axis=0)

    print(f"Loaded successfully! Lattice Sites: {num_sites} | (Lx={Lx}, Ly={Ly})")
    return X_data, y_psi, observables, param_vector, num_sites, Lx, Ly


# ==========================================
# 3. GNN MODULES (SIZE-AGNOSTIC)
# ==========================================
class GCNLayer(nn.Module):
    """Message passing layer aggregating node representations over the Kagome graph."""
    features: int

    @nn.compact
    def __call__(self, node_feats, adj_matrix):
        x = nn.Dense(self.features)(node_feats)
        x = jnp.matmul(adj_matrix, x)  # Neighbor aggregation
        return nn.tanh(x)


class NodePropertyHead(nn.Module):
    """Shared per-node MLP predicting 1D site vector properties (e.g. charge density)."""
    hidden_dim: int = 64

    @nn.compact
    def __call__(self, node_feats):
        x = nn.Dense(self.hidden_dim)(node_feats)
        x = nn.tanh(x)
        x = nn.Dense(1)(x)
        return jnp.squeeze(x, axis=-1)


class BilinearCorrelationHead(nn.Module):
    """Size-agnostic head generating symmetric N x N correlation matrices."""
    @nn.compact
    def __call__(self, node_feats):
        feat_dim = node_feats.shape[-1]
        W = self.param('W', nn.initializers.normal(stddev=0.1), (feat_dim, feat_dim))
        
        # Tensor contraction H * W * H^T
        raw_corr = jnp.einsum('bni,ij,bmj->bnm', node_feats, W, node_feats)
        
        # Enforce C_ij = C_ji
        return 0.5 * (raw_corr + jnp.swapaxes(raw_corr, 1, 2))


class KagomeGNN_NQS(nn.Module):
    """Unified size-agnostic GNN architecture."""
    hidden_dim: int = 128
    head_dim: int = 64

    @nn.compact
    def __call__(self, x_input, adj_matrix):
        node_feats = jnp.expand_dims(x_input, axis=-1)
        
        # Message passing trunk
        h = GCNLayer(self.hidden_dim)(node_feats, adj_matrix)
        h = GCNLayer(self.hidden_dim)(h, adj_matrix)

        # Head 0: Quantum State Amplitudes log|psi|
        h_global = jnp.mean(h, axis=1)
        log_psi = jnp.squeeze(nn.Dense(1)(h_global), axis=-1)

        # Observable Prediction Heads
        pred_heads = {
            "charge_density": NodePropertyHead(hidden_dim=self.head_dim)(h),
            "spin_corr": BilinearCorrelationHead()(h),
            "charge_corr": BilinearCorrelationHead()(h),
            "double_occupancy": NodePropertyHead(hidden_dim=self.head_dim)(h)
            #"spin_structure_factor",
            #"charge_structure_factor"
        }

        return log_psi, pred_heads


# ==========================================
# 4. TRAINING LOOP
# ==========================================
def train_kagome_nqs(
    X_data, A_data, y_psi, observables_true, params, opt_state, optimizer, model, num_epochs=1500, w_psi=1.0
):
    print("--- Training Size-Agnostic GNN NQS ---")

    targets_jax = {k: jnp.array(v)[None, ...] for k, v in observables_true.items()}
    X_inputs = jnp.array(X_data)
    A_inputs = jnp.repeat(A_data[None, ...], X_inputs.shape[0], axis=0)
    y_targets = jnp.array(y_psi) if y_psi is not None else None

    @jax.jit
    def loss_fn(params, x_batch, adj_batch, y_batch):
        log_psi, pred_heads = model.apply({"params": params}, x_batch, adj_batch)

        # Overlap Loss
        if y_batch is not None:
            log_psi_shifted = log_psi - jnp.max(log_psi)
            psi = jnp.exp(log_psi_shifted)
            overlap = jnp.dot(psi, y_batch)
            norm_model = jnp.linalg.norm(psi)
            norm_true = jnp.linalg.norm(y_batch)
            loss_psi = 1.0 - ((overlap**2) / (norm_model**2 * norm_true**2 + 1e-12))
        else:
            loss_psi = 0.0

        # Physical Properties MSE Loss
        total_obs_loss = 0.0
        head_losses = {}
        for name, target in targets_jax.items():
            l_head = jnp.mean((pred_heads[name] - target) ** 2)
            head_losses[name] = l_head
            total_obs_loss += 0.5 * l_head

        total_loss = w_psi * loss_psi + total_obs_loss
        return total_loss, (loss_psi, head_losses, pred_heads)

    @jax.jit
    def train_step(params, opt_state, x_batch, adj_batch, y_batch):
        (total_loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(
            params, x_batch, adj_batch, y_batch
        )
        updates, new_opt_state = optimizer.update(grads, opt_state)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, total_loss, aux

    for epoch in range(num_epochs):
        print(f"[{epoch+1}/{num_epochs}]")
        params, opt_state, total_loss, aux = train_step(params, opt_state, X_inputs, A_inputs, y_targets)
        loss_psi, head_losses, pred_heads = aux
        #if (epoch + 1) % 10 == 0:
         #   loss_str = " | ".join([f"{k[:6]}: {v:.2e}" for k, v in head_losses.items()])
         #   print(f"Epoch {epoch+1:4d} | Total Loss: {total_loss:.6f} | {loss_str}")
            
        if (epoch + 1) % 10 == 0:
            loss_str = " | ".join([f"{k[:6]}: {v:.2e}" for k, v in head_losses.items()])
            print(f"Epoch {epoch+1:4d} | Total Loss: {total_loss:.6f} | {loss_str}")

    pred_heads_squeezed = {k: np.array(v[0]) for k, v in pred_heads.items()}
    return params, opt_state, pred_heads_squeezed, head_losses


# ==========================================
# 5. ENTRY POINT
# ==========================================
if __name__ == "__main__":
    model = KagomeGNN_NQS(hidden_dim=128, head_dim=64)
    rng = jax.random.PRNGKey(42)

    # Initialize GNN weights using a standard 3-site dummy graph
    dummy_x = jnp.ones((1, 3))
    dummy_a = jnp.ones((1, 3, 3))
    params = model.init(rng, dummy_x, dummy_a)["params"]

    num_samples = 10
    num_cycles = 5
    num_epochs = 40
    total_steps = num_samples * num_cycles * num_epochs
    
    schedule = optax.cosine_decay_schedule(init_value=1e-4, decay_steps=total_steps, alpha=0.01)
    #optimizer = optax.chain(
    #    optax.clip_by_global_norm(1.0),  # Prevents e-01 loss spikes between files
    #    optax.adam(learning_rate=schedule)
    #)
    optimizer = optax.adam(learning_rate=schedule)
    opt_state = optimizer.init(params)

    # Sequence of HDF5 files representing different state points or lattice sizes
    #data_files = ["three_site_Kagome_Hubbard_296.h5"]
    
    print("Model Type: ", type(model))
    for j in range(num_cycles):
        for i in range(num_samples):
            h5_file = f"data_outputs/wavefunctions/six_site_kagome_{i:03d}.h5"
            if not os.path.exists(h5_file):
                print(f"Skipping missing file: {h5_file}")
                continue
    
            print(f"\n{'='*65}\nProcessing File: {h5_file} | Cycle: {j+1}/{num_cycles}\n{'='*65}")
    
            # 1. Load Data
            X_data, y_psi, observables_true, param_vector, num_sites, Lx, Ly = load_kagome_dataset(h5_file)
    
            # 2. Build Exact 2D Kagome Adjacency Matrix
            A_data = build_kagome_adjacency(Lx=Lx, Ly=Ly, periodic=True)
    
            # 3. Train/Transfer Model Weights Across Datasets
            params, opt_state, pred_heads, head_losses = train_kagome_nqs(
                X_data, A_data, y_psi, observables_true, params, opt_state, optimizer, model, num_epochs=num_epochs
            )
    for j in range(num_cycles):
        for i in range(num_samples):
            h5_file = f"data_outputs/wavefunctions/three_site_kagome_{i:03d}.h5"
            if not os.path.exists(h5_file):
                print(f"Skipping missing file: {h5_file}")
                continue
    
            print(f"\n{'='*65}\nProcessing File: {h5_file} | Cycle: {j}/{num_cycles}\n{'='*65}")
    
            # 1. Load Data
            X_data, y_psi, observables_true, param_vector, num_sites, Lx, Ly = load_kagome_dataset(h5_file)
    
            # 2. Build Exact 2D Kagome Adjacency Matrix
            A_data = build_kagome_adjacency(Lx=Lx, Ly=Ly, periodic=True)
    
            # 3. Train/Transfer Model Weights Across Datasets
            params, opt_state, pred_heads, head_losses = train_kagome_nqs(
                X_data, A_data, y_psi, observables_true, params, opt_state, optimizer, model, num_epochs=num_epochs
            )
    for j in range(num_cycles):
        for i in range(num_samples):
            j = i + 20
            h5_file = f"data_outputs/wavefunctions/three_site_kagome_{j:03d}.h5"
            if not os.path.exists(h5_file):
                print(f"Skipping missing file: {h5_file}")
                continue
    
            print(f"\n{'='*65}\nProcessing File: {h5_file} | Cycle: {j}/{num_cycles}\n{'='*65}")
    
            # 1. Load Data
            X_data, y_psi, observables_true, param_vector, num_sites, Lx, Ly = load_kagome_dataset(h5_file)
    
            # 2. Build Exact 2D Kagome Adjacency Matrix
            A_data = build_kagome_adjacency(Lx=Lx, Ly=Ly, periodic=True)
    
            # 3. Train/Transfer Model Weights Across Datasets
            params, opt_state, pred_heads, head_losses = train_kagome_nqs(
                X_data, A_data, y_psi, observables_true, params, opt_state, optimizer, model, num_epochs=num_epochs
            )
        # 4. Display Results
        #print("\n[Final Mean Squared Errors]:")
        #for k, v in head_losses.items():
        #    print(f"  - Head '{k}': MSE = {float(v):.4e}")

        #if "spin_corr" in pred_heads:
        #    print("\n[Predicted Symmetrized Spin Correlation Matrix (3x3 Block)]:")
       #     print(np.array2string(pred_heads["spin_corr"][:3, :3], precision=4, suppress_small=True))
            

Model Type:  <class '__main__.KagomeGNN_NQS'>

Processing File: data_outputs/wavefunctions/six_site_kagome_000.h5 | Cycle: 1/5
--- Loading Kagome Dataset from 'data_outputs/wavefunctions/six_site_kagome_000.h5' ---


C:\Users\chait\AppData\Local\Temp\ipykernel_16984\2378912713.py:121: ComplexWarning: Casting complex values to real discards the imaginary part
  y_psi = np.array(psi_dense, dtype=np.float32).reshape(-1)


Loaded successfully! Lattice Sites: 6 | (Lx=2, Ly=1)
--- Training Size-Agnostic GNN NQS ---
[1/40]
[2/40]
[3/40]
[4/40]
[5/40]
[6/40]
[7/40]
[8/40]
[9/40]
[10/40]
Epoch   10 | Total Loss: 2.214031 | charge: 2.58e-01 | charge: 2.26e-01 | double: 3.63e-02 | spin_c: 1.91e+00
[11/40]
[12/40]
[13/40]
[14/40]
[15/40]
[16/40]
[17/40]
[18/40]
[19/40]
[20/40]
Epoch   20 | Total Loss: 6.650912 | charge: 1.09e+01 | charge: 2.05e-01 | double: 3.41e-02 | spin_c: 1.41e-01
[21/40]
[22/40]
[23/40]
[24/40]
[25/40]
[26/40]
[27/40]
[28/40]
[29/40]
[30/40]
Epoch   30 | Total Loss: 1.738515 | charge: 9.48e-01 | charge: 1.77e-01 | double: 2.07e-02 | spin_c: 3.31e-01
[31/40]
[32/40]
[33/40]
[34/40]
[35/40]
[36/40]
[37/40]
[38/40]
[39/40]
[40/40]
Epoch   40 | Total Loss: 1.668637 | charge: 9.93e-01 | charge: 1.56e-01 | double: 1.15e-02 | spin_c: 1.77e-01

Processing File: data_outputs/wavefunctions/six_site_kagome_001.h5 | Cycle: 1/5
--- Loading Kagome Dataset from 'data_outputs/wavefunctions/six_site_kagome_

In [2]:
import jax
import jax.numpy as jnp
import netket as nk
import flax
from transformers import FlaxAutoModel

#required to prevent checkpointing conflicts with HuggingFace (?)
flax.config.update('flax_use_orbax_checkpointing', False)

#download and load pre-trained model
print("Loading pre-trained model\n...")
p = 0.7 #* fix the value of the external field
L = 6
revision = f"L{L}_p{p}"
trial_model = FlaxAutoModel.from_pretrained("nqs-models/heisenberg_disorder_fnqs", trust_remote_code = True, revision=revision)
modify_model = FlaxAutoModel.from_pretrained("nqs-models/heisenberg_disorder_fnqs", trust_remote_code = True, revision=revision)

print("Define physical system (for testing)\n...")

#Standard FNQS models are often trained on 10x10 grids (100 spins), much greater than our samples??
#10x10 square lattice with periodic boundary conditions
graph = nk.graph.Square(10,pbc=True)
#print(f"\tHelp: nk.graph: {help(nk.graph)}\n")
hi = nk.hilbert.Spin(s=0.5, N=graph.n_nodes)

print("Define Hamiltonian\n...")

#create Heisenberg hamiltonian
H = nk.operator.Heisenberg(hilbert=hi, graph=graph)

print("Initialize Monte Carlo State\n...")

#set up Markov Chain Monte Carlo sampler that preserves total magnetization
sampler = nk.sampler.MetropolisExchange(hilbert=hi, graph=graph)

#wrap the model in a netket variational state
#critical: we inject the pre-trained weights using the 'variables' argument
vstate = nk.vqs.MCState(
    sampler = sampler,
    model=trial_model,
    n_samples=1024,
    variables={'params':trial_model.params}
)

print("Running Evaluation\n...")

energy = vstate.expect(H)

print(f"\nVerification Complete:\n\tEnergy Expectation Value: {energy}")


Loading pre-trained model
...


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Define physical system (for testing)
...
Define Hamiltonian
...
Initialize Monte Carlo State
...


TypeError: ViTFNQSModel.__call__() missing 1 required positional argument: 'coups'

In [2]:
help(nk.operator.Ising)

Help on class IsingJax in module netket.operator._ising.jax:

class IsingJax(netket.operator._ising.base.IsingBase, netket.operator._discrete_operator_jax.DiscreteJaxOperator)
 |  IsingJax(hilbert: netket.hilbert.abstract_hilbert.AbstractHilbert, graph: Union[netket.graph.abstract_graph.AbstractGraph, numpy.ndarray, jax.jaxlib._jax.Array], h: float, J: float, dtype: Union[NoneType, str, type[Any], numpy.dtype, netket.utils.types._SupportsDType])
 |  
 |  Jax-based implementation of the Transverse-Field Ising Hamiltonian
 |  :math:`-h\sum_i \sigma_i^{(x)} +J\sum_{\langle i,j\rangle} \sigma_i^{(z)}\sigma_j^{(z)}`.
 |  
 |  This implementation is considerably faster than the
 |  Ising hamiltonian constructed by summing
 |  :class:`~netket.operator.LocalOperator` s.
 |  
 |  Method resolution order:
 |      IsingJax
 |      netket.operator._ising.base.IsingBase
 |      netket.operator._hamiltonian.SpecialHamiltonian
 |      netket.operator._discrete_operator_jax.DiscreteJaxOperator
 |     

In [ ]:
from functools import partial
import jax
import jax.numpy as jnp
import netket as nk
import math
import flax
from flax.training import checkpoints
import numpy as np
from netket.operator.spin import sigmax, sigmaz, sigmay

flax.config.update('flax_use_orbax_checkpointing', False)

p = 0.7 #* fix the value of the external field
L = 6
revision = f"L{L}_p{p}"

def edges_square_lattice(L):
    Ns = L*L
    indices = np.arange(Ns)
    indices_right = (indices+1)%L + L*(indices//L)
    indices_down = (indices+L)%Ns
    first = np.c_[indices, indices_right]
    second = np.c_[indices, indices_down]

    edges = np.concatenate([first, second], axis=0)
    return edges

def coupling_heis_random(random_J, edges):
    edges_with_random_vars = list(zip(edges, random_J))
    return edges_with_random_vars

def si_sj(hi, i, j, txy=1.0):
    # 0.25 factor is to take into account for spin operators
    return 0.25*(txy * (sigmax(hi, i) * sigmax(hi, j) + sigmay(hi, i) * sigmay(hi, j)) + sigmaz(hi, i) * sigmaz(hi, j))

def heisenberg_hamiltonian(edges_Js, hi, txy=1.0):
    ham = 0.0
    for (ij, J) in edges_Js:
        ham += J * si_sj(hi,  ij[0], ij[1], txy)
    return ham

from transformers import FlaxAutoModel
wf = FlaxAutoModel.from_pretrained("nqs-models/heisenberg_disorder_fnqs", 
                                   trust_remote_code=True, 
                                   revision=revision)

N_params = nk.jax.tree_size(wf.params)
print('Number of parameters = ', N_params, flush=True)

lattice = nk.graph.Hypercube(length=L, n_dim=2, pbc=True)
hilbert = nk.hilbert.Spin(s=1/2, N=lattice.n_nodes, total_sz=0)

# Random Heisenberg Hamiltonian
from huggingface_hub import hf_hub_download
coups_path = hf_hub_download(repo_id="nqs-models/heisenberg_disorder_fnqs", filename="coups", revision=revision)
random_J = np.loadtxt(coups_path)[0]
edges = edges_square_lattice(L)
edges_Js = coupling_heis_random(random_J=random_J, edges=edges)

N_mc = 6000

hamiltonian = heisenberg_hamiltonian(edges_Js, hilbert)
sampler = nk.sampler.MetropolisExchange(hilbert=hilbert,
                                        graph=lattice,
                                        d_max=2,
                                        n_chains=N_mc,
                                        sweep_size=lattice.n_nodes)

key = jax.random.key(0)
key, subkey = jax.random.split(key, 2)
vstate = nk.vqs.MCState(sampler=sampler, 
                        apply_fun=partial(wf.__call__, coups=random_J), 
                        sampler_seed=subkey,
                        n_samples=N_mc, 
                        n_discard_per_chain=0,
                        variables=wf.params,
                        chunk_size=N_mc)

path = hf_hub_download(repo_id="nqs-models/heisenberg_disorder_fnqs", filename="spins", revision=revision)
samples = checkpoints.restore_checkpoint(path, target=None)
samples = jnp.array(samples, dtype='int8')
vstate.sampler_state = vstate.sampler_state.replace(σ = samples)

import time
# Sample the model
for _ in range(10):
    start = time.time()
    E = vstate.expect(hamiltonian)
    vstate.sample()

    print("Mean: ", E.mean.real / lattice.n_nodes, "\t time=", time.time()-start, flush=True)


C:\Users\chait\AppData\Local\hermes\hermes-agent\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chait\.cache\huggingface\hub\models--nqs-models--heisenberg_disorder_fnqs. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The argument `trust_remote_code` is to be used with Auto classes. It has no eff

Number of parameters =  793240


C:\Users\chait\AppData\Local\hermes\hermes-agent\venv\Lib\site-packages\netket\operator\_local_operator\base.py:348: OperatorMultiplicationDeprecationWarning: 
The '*' operator for multiplying operators is deprecated and will be removed in a future version.

Please use the '@' operator instead:
  - Replace: operator1 * operator2
  - With:    operator1 @ operator2

The '@' operator is Python's standard matrix multiplication operator.


-------------------------------------------------------
For more detailed informations, visit the following link:
	 https://netket.readthedocs.io/en/latest/api/_generated/errors/netket.errors.OperatorMultiplicationDeprecationWarning.html
or the list of all common errors and warnings at
	 https://netket.readthedocs.io/en/latest/api/errors.html
-------------------------------------------------------

  warnings.warn(OperatorMultiplicationDeprecationWarning())
C:\Users\chait\AppData\Local\hermes\hermes-agent\venv\Lib\site-packages\netket\operator\_local_oper

Mean:  

In [1]:
import os
import h5py
import numpy as np
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax

# ==========================================
# 1. KAGOME ADJACENCY MATRIX GENERATOR
# ==========================================
def build_kagome_adjacency(Lx: int, Ly: int, periodic: bool = True) -> jnp.ndarray:
    """Builds the exact 2D Kagome lattice adjacency matrix."""
    num_unit_cells = Lx * Ly
    num_sites = 3 * num_unit_cells
    adj = np.zeros((num_sites, num_sites), dtype=np.float32)

    def get_site_index(x, y, sublattice):
        if periodic:
            x = x % Lx
            y = y % Ly
        else:
            if x < 0 or x >= Lx or y < 0 or y >= Ly:
                return -1
        uc_index = x + y * Lx
        return 3 * uc_index + sublattice

    def add_edge(site1, site2):
        if site1 != -1 and site2 != -1:
            adj[site1, site2] = 1.0
            adj[site2, site1] = 1.0

    for y in range(Ly):
        for x in range(Lx):
            A = get_site_index(x, y, 0)
            B = get_site_index(x, y, 1)
            C = get_site_index(x, y, 2)

            # Intra-unit cell
            add_edge(A, B)
            add_edge(B, C)
            add_edge(C, A)

            # Inter-unit cell
            B_left = get_site_index(x - 1, y, 1)
            add_edge(A, B_left)
            A_down = get_site_index(x, y - 1, 0)
            add_edge(C, A_down)
            C_diag = get_site_index(x - 1, y - 1, 2)
            add_edge(B, C_diag)

    return jnp.array(adj)


# ==========================================
# 2. HDF5 DATASET LOADER (SURROGATE VERSION)
# ==========================================
def load_kagome_dataset(h5_filepath):
    print(f"--- Loading Kagome Dataset from '{h5_filepath}' ---")

    if not os.path.exists(h5_filepath):
        raise FileNotFoundError(f"Could not locate file: {h5_filepath}")

    with h5py.File(h5_filepath, "r") as f:
        p = f["model_params"]
        Lx = int(p["Lx"][()]) if "Lx" in p else 1
        Ly = int(p["Ly"][()]) if "Ly" in p else 1

        U = float(p["U"][()]) if "U" in p else 0.0
        t1 = float(p["t1"][()]) if "t1" in p else 1.0
        t2 = float(p["t2"][()]) if "t2" in p else 0.0
        theta_x = float(p["theta_x"][()]) if "theta_x" in p else 0.0
        theta_y = float(p["theta_y"][()]) if "theta_y" in p else 0.0

        param_vector = np.array([U, t1, t2, theta_x, theta_y], dtype=np.float32)
        obs = f["observables"]

        def get_obs(key):
            if key in obs:
                return np.array(obs[key], dtype=np.float32)
            raise KeyError(f"Required observable '{key}' not found in HDF5 file.")

        density_true = np.squeeze(get_obs("charge_density"))
        spin_corr_true = np.squeeze(get_obs("spin_corr"))
        
        # Try to load energy; default to 0.0 if not present for testing
        if "energy" in f:
            energy_true = float(f["energy"][()])
        else:
            energy_true = 0.0

        num_sites = len(density_true)

        observables = {
            "charge_density": density_true,
            "spin_corr": spin_corr_true,
            "energy": np.array([energy_true], dtype=np.float32)
        }

    # Surrogate Input: Params + One-Hot Sublattice ID
    X_params = np.repeat(param_vector[None, :], num_sites, axis=0)
    
    # Create one-hot vectors for sublattices A, B, C (0, 1, 2)
    sublattice_ids = np.eye(3, dtype=np.float32)[np.arange(num_sites) % 3]

    # Surrogate Input: Every node gets the [U, t1, t2, theta_x, theta_y] parameter vector
    X_data = np.concatenate([X_params, sublattice_ids], axis=1)

    print(f"Loaded successfully! Lattice Sites: {num_sites} | (Lx={Lx}, Ly={Ly})")
    return X_data, observables, param_vector, num_sites, Lx, Ly


# ==========================================
# 3. GNN MODULES (SURROGATE)
# ==========================================
class GCNLayer(nn.Module):
    features: int

    @nn.compact
    def __call__(self, node_feats, adj_matrix):
        x = nn.Dense(self.features)(node_feats)
        x = jnp.matmul(adj_matrix, x)
        return nn.silu(x)  # Changed to silu for better gradient flow


class NodePropertyHead(nn.Module):
    hidden_dim: int = 64

    @nn.compact
    def __call__(self, node_feats):
        x = nn.Dense(self.hidden_dim)(node_feats)
        x = nn.silu(x)
        x = nn.Dense(1)(x)
        return jnp.squeeze(x, axis=-1)


class BilinearCorrelationHead(nn.Module):
    @nn.compact
    def __call__(self, node_feats):
        feat_dim = node_feats.shape[-1]
        W = self.param('W', nn.initializers.normal(stddev=0.1), (feat_dim, feat_dim))
        raw_corr = jnp.einsum('bni,ij,bmj->bnm', node_feats, W, node_feats)
        return 0.5 * (raw_corr + jnp.swapaxes(raw_corr, 1, 2))


class GlobalPropertyHead(nn.Module):
    """Pools graph features to predict a single macroscopic scalar (e.g., Energy)."""
    hidden_dim: int = 64

    @nn.compact
    def __call__(self, node_feats):
        global_feat = jnp.sum(node_feats, axis=1) 
        x = nn.Dense(self.hidden_dim)(global_feat)
        x = nn.silu(x)
        x = nn.Dense(1)(x)
        return x  # <-- REMOVED squeeze here. Shape is now (Batch, 1)


class KagomeGNN_Surrogate(nn.Module):
    """Parameter-to-Observable Surrogate Model."""
    hidden_dim: int = 128
    head_dim: int = 64

    @nn.compact
    def __call__(self, x_input, adj_matrix):
        # x_input shape: (batch, num_sites, 5)
        h = GCNLayer(self.hidden_dim)(x_input, adj_matrix)
        h = GCNLayer(self.hidden_dim)(h, adj_matrix)

        pred_heads = {
            "charge_density": NodePropertyHead(hidden_dim=self.head_dim)(h),
            "spin_corr": BilinearCorrelationHead()(h),
            "energy": GlobalPropertyHead(hidden_dim=self.head_dim)(h)
        }

        return pred_heads


# ==========================================
# 4. TRAINING LOOP (SURROGATE)
# ==========================================
def train_kagome_surrogate(X_data, A_data, observables_true, params, opt_state, optimizer, model, num_epochs=600):
    print("--- Training Surrogate Parameter-to-Observable GNN ---")

    targets_jax = {k: jnp.array(v) for k, v in observables_true.items()}
    X_inputs = jnp.array(X_data)
    A_inputs = jnp.array(A_data)

    # Automatically compute loss weights based on the inverse variance of each target
    target_variances = {k: jnp.var(v) + 1e-6 for k, v in targets_jax.items()}
    loss_weights = {k: 1.0 / v for k, v in target_variances.items()}

    print("\n[Automatic Loss Weights (Inverse Variance)]:")
    for k, w in loss_weights.items():
        print(f"  - {k}: {float(w):.4f}")
    print()

    @jax.jit
    def loss_fn(params, x_batch, adj_batch):
        pred_heads = model.apply({"params": params}, x_batch, adj_batch)

        total_obs_loss = 0.0
        head_losses = {}
        for name, target in targets_jax.items():
            l_head = jnp.mean((pred_heads[name] - target) ** 2)
            head_losses[name] = l_head
            
            # Apply our computed weights to balance the scales!
            total_obs_loss += l_head * loss_weights[name]

        return total_obs_loss, (head_losses, pred_heads)

    @jax.jit
    def train_step(params, opt_state, x_batch, adj_batch):
        (total_loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(
            params, x_batch, adj_batch
        )
        updates, new_opt_state = optimizer.update(grads, opt_state)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, total_loss, aux

    for epoch in range(num_epochs):
        params, opt_state, total_loss, aux = train_step(params, opt_state, X_inputs, A_inputs)
        head_losses, pred_heads = aux

        if (epoch + 1) % 150 == 0:
            loss_str = " | ".join([f"{k[:6]}: {float(v):.2e}" for k, v in head_losses.items()])
            print(f"Epoch {epoch+1:4d} | Weighted Total Loss: {float(total_loss):.4f} | {loss_str}")

    # Return full batched predictions so we can print them individually
    pred_heads_numpy = {k: np.array(v) for k, v in pred_heads.items()}
    return params, opt_state, pred_heads_numpy, head_losses


# ==========================================
# 5. ENTRY POINT
# ==========================================
if __name__ == "__main__":
    # 1. ACCUMULATE ALL DATA FOR BATCHING
    print("Gathering dataset...")
    all_X, all_A = [], []
    all_obs = {"charge_density": [], "spin_corr": [], "energy": []}
    loaded_files = [] 

    for i in range(100):
        h5_file = r"C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunctions\three_site_kagome_"+f"{i:03d}.h5"
        try:
            X_data, obs_true, param_vec, num_sites, Lx, Ly = load_kagome_dataset(h5_file)
            A_data = build_kagome_adjacency(Lx=Lx, Ly=Ly, periodic=True)

            all_X.append(X_data)
            all_A.append(A_data)
            for k in all_obs.keys():
                all_obs[k].append(obs_true[k])
            loaded_files.append(i)
        except FileNotFoundError:
            continue

    batch_X = np.stack(all_X, axis=0)
    batch_A = np.stack(all_A, axis=0)
    batch_obs = {k: np.stack(v, axis=0) for k, v in all_obs.items()}

    print(f"\nDataset batched! Total files loaded: {batch_X.shape[0]}")
    print(f"Batched Input Shape: {batch_X.shape} -> (Batch, Nodes, Features)\n")

    # ==========================================
    # NEW: Z-SCORE NORMALIZATION
    # ==========================================
    print("Applying Z-score normalization...")
    
    # 1. Normalize Inputs (batch_X)
    X_mean = np.mean(batch_X, axis=(0, 1), keepdims=True)
    X_std = np.std(batch_X, axis=(0, 1), keepdims=True) + 1e-8
    batch_X_norm = (batch_X - X_mean) / X_std

    # 2. Normalize Targets (batch_obs)
    obs_stats = {}
    batch_obs_norm = {}
    for k, v in batch_obs.items():
        # Compute mean and std across the batch dimension
        mean_val = np.mean(v, axis=0, keepdims=True)
        std_val = np.std(v, axis=0, keepdims=True) + 1e-8
        obs_stats[k] = {'mean': mean_val, 'std': std_val}
        
        # Create normalized target arrays
        batch_obs_norm[k] = (v - mean_val) / std_val
    # ==========================================

    # 2. INITIALIZE MODEL
    model = KagomeGNN_Surrogate(hidden_dim=128, head_dim=64)
    rng = jax.random.PRNGKey(42)

    dummy_x = jnp.ones((1, 3, 8)) 
    dummy_a = jnp.ones((1, 3, 3))
    params = model.init(rng, dummy_x, dummy_a)["params"]

    schedule = optax.cosine_decay_schedule(init_value=1e-3, decay_steps=30000, alpha=0.01)
    optimizer = optax.adam(learning_rate=schedule)
    opt_state = optimizer.init(params)

    # 3. TRAIN ON THE NORMALIZED DATASET
    # Notice we pass batch_X_norm and batch_obs_norm here
    for i in range(600):
        params, opt_state, pred_heads_norm, head_losses = train_kagome_surrogate(
            batch_X_norm, batch_A, batch_obs_norm, params, opt_state, optimizer, model, num_epochs=1
        )

  
    # ==========================================
    # NEW: UN-NORMALIZE PREDICTIONS
    # ==========================================
    # We must scale the predictions back up to real physical units for printing
    pred_heads = {}
    for k in pred_heads_norm.keys():
        mean_val = np.squeeze(obs_stats[k]['mean'])
        std_val = np.squeeze(obs_stats[k]['std'])
        pred_heads[k] = (pred_heads_norm[k] * std_val) + mean_val
    # ==========================================

    # 4. DISPLAY RESULTS
    print("\nRESULTS PER FILE (True vs Predicted in Physical Units)")
    print("=======================================================")
    
    num_files = batch_X.shape[0]
    for i in range(num_files):
        file_idx = loaded_files[i]
        print(f"--- File three_site_kagome_{file_idx:03d}.h5 ---")
        
        # Safely extract and cast to float
        e_true = float(np.squeeze(batch_obs['energy'][i]))
        e_pred = float(np.squeeze(pred_heads['energy'][i]))
        print(f"  Energy:        True = {e_true:8.4f} | Pred = {e_pred:8.4f}")
        
        c_true = float(np.squeeze(batch_obs['charge_density'][i][0]))
        c_pred = float(np.squeeze(pred_heads['charge_density'][i][0]))
        print(f"  Charge (site0): True = {c_true:8.4f} | Pred = {c_pred:8.4f}")
        
        print("  Spin Matrix (True):")
        for row in batch_obs['spin_corr'][i]:
            print("    " + " ".join([f"{val:8.4f}" for val in row]))
            
        print("  Spin Matrix (Pred):")
        for row in pred_heads['spin_corr'][i]:
            print("    " + " ".join([f"{val:8.4f}" for val in row]))
            
        print("-" * 55)

Gathering dataset...
--- Loading Kagome Dataset from 'C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunctions\three_site_kagome_000.h5' ---
--- Loading Kagome Dataset from 'C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunctions\three_site_kagome_001.h5' ---
--- Loading Kagome Dataset from 'C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunctions\three_site_kagome_002.h5' ---
--- Loading Kagome Dataset from 'C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunctions\three_site_kagome_003.h5' ---
--- Loading Kagome Dataset from 'C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunctions\three_site_kagome_004.h5' ---
--- Loading Kagome Dataset from 'C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunctions\three_site_kagome_005.h5' ---
--- Loading Kagome Dataset from 'C:\Users\jj9ba\Research\DREU QIS-AI\kagome-datasets\kagome-datasets\wavefunc

ValueError: need at least one array to stack